# 미국 대형주 HRL 일반화 검증 — 제3의 시장 (`paper/DRAFT.md`/`DRAFT_KR.md` 7장 대응)

크립토(최종 배포)·KOSPI(§7, 부정적 결과)에 이어 **세 번째로 독립적인 시장**에서 이 프로젝트의 두 핵심
발견 — (1) 순수 단일 정책 RL의 실패, (2) Q-teacher 없는 β-조건부 전문가 풀 + 추론 시점 스무딩(라우터
없음)이라는 최종 레시피의 valid/test 일관성 — 을 검증한다.

**범위 축소(중요, 반드시 먼저 읽을 것)**: KOSPI·크립토는 Chronos+MTGNN 임베딩 백본을 몇 달에 걸쳐
구축한 뒤 그 위에 RL을 얹었다(크립토: 4월 데이터 수집~8월 백본 비교). 동일한 백본을 미국 주식에
새로 구축하는 것은 그 자체로 별도의 몇 달짜리 프로젝트라 이번 범위에서 제외한다. 대신 이 노트북은
**EIIE/LSRE-CAAN 베이스라인 노트북이 이미 쓰고 검증한 "원시 가격 윈도우" 상태 표현**(임베딩 불필요,
close/high/low 비율 윈도우 + 직전 비중)을 그대로 재사용해 임베딩 백본 구축 단계를 완전히 생략한다.
이는 성능을 극대화하기 위한 설계가 아니라 **세 번째 시장에서 방법론(단일정책 실패 재현 여부, β풀
다양성, valid/test 일관성 진단)이 통하는지를 빠르게 검증하기 위한 의도적 단순화**다 — 크립토/KOSPI
결과와 절대 성능을 직접 비교하지 않는다(입력 표현 자체가 다르므로).

**데이터**: `yfinance`로 받는 S&P 500 대형주 44종목(크립토·KOSPI HRL 실험과 유사한 유니버스 규모,
11개 GICS 섹터에 걸쳐 선정) 일별 OHLC, 2015–2025.

**아키텍처**: `crypto_eiie_baseline_colab.ipynb`의 `PriceWindowPortfolioEnv`(원시 가격 윈도우 + 현금
레그, softmax 비중)와 EIIE 스타일 CNN 특징 추출기를 그대로 재사용 — 새 네트워크를 만들지 않는다.
Stage A(레짐분할)·Stage B(β=-90/-30/30/90 풀, 우선순위 샘플링, 라우터 없음)·추론 시점 스무딩은
`kospi_hrl_generalization_colab.ipynb`와 동일한 축소 설계(top-K/게이트·라우터 제외)를 따른다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yfinance stable-baselines3[extra] gymnasium pyarrow


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 16.5 MB/s eta 0:00:00


## 1. 설정

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

import yfinance as yf

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data' / 'us_equity'
MODEL_DIR  = DRIVE_ROOT / 'models' / 'us_equity_hrl'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# S&P 500 대형주 44종목 — 11개 GICS 섹터 분산, 2015년 이전 상장(전체 구간 커버) 종목만 선정
UNIVERSE = [
    # 정보기술 (9)
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'ORCL', 'CSCO', 'INTC',
    # 금융 (7)
    'JPM', 'BAC', 'WFC', 'GS', 'MS', 'AXP', 'BLK',
    # 헬스케어 (6)
    'JNJ', 'UNH', 'PFE', 'MRK', 'ABBV', 'TMO',
    # 필수·임의소비재 (8)
    'PG', 'KO', 'PEP', 'WMT', 'HD', 'MCD', 'NKE', 'SBUX',
    # 산업재 (6)
    'BA', 'CAT', 'GE', 'HON', 'UPS', 'MMM',
    # 에너지 (2)
    'XOM', 'CVX',
    # 유틸리티 (2)
    'NEE', 'DUK',
    # 커뮤니케이션 (3)
    'VZ', 'T', 'DIS',
    # 소재 (1)
    'LIN',
]
N_ASSETS_TARGET = len(UNIVERSE)
print(f'유니버스: {N_ASSETS_TARGET}종목')

TC = 0.001   # 크립토·KOSPI와 동일 관례(10bp) — 미국 대형주 실제 리테일 수수료는 이보다 낮은 경우가
             # 많지만(예: 무료 브로커리지), 세 시장 비교의 일관성을 위해 동일 값 유지
WINDOW     = 50   # EIIE/LSRE-CAAN 노트북과 동일
N_CHANNELS = 3    # [close, high, low] / 마지막 종가 비율

TRAIN_START = '2015-01-01'
TRAIN_END   = '2022-12-31'
VALID_START = '2023-01-01'
VALID_END   = '2023-12-31'
TEST_START  = '2024-01-01'
TEST_END    = '2025-12-31'

N_LABELS       = 5
RISK_THRESHOLD = 0.2
LOWPASS_WINDOW = 20    # KOSPI와 동일(일봉 기준 ~1개월 거래일)
POOL_BETAS          = [-90, -30, 30, 90]
POOL_TIMESTEPS_EACH = 100_000   # 크립토·KOSPI와 동일 예산
SINGLE_PPO_TIMESTEPS = 100_000  # §동기 확인용 단일 정책 베이스라인도 동일 예산
SMOOTH_ALPHAS        = [1.0, 0.3, 0.15, 0.05, 0.03, 0.02, 0.01]

print('설정 완료')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

유니버스: 44종목
설정 완료


## 2. 데이터 다운로드 (yfinance) — 실패 종목은 자동 제외

In [3]:
%%time
CACHE_PATH = DATA_DIR / 'ohlc_raw.parquet'

if CACHE_PATH.exists():
    print('캐시 로드 중...')
    raw = pd.read_parquet(CACHE_PATH)
else:
    print(f'{len(UNIVERSE)}종목 다운로드 중 (yfinance)...')
    raw = yf.download(
        UNIVERSE, start=TRAIN_START, end=TEST_END,
        auto_adjust=True, group_by='ticker', progress=True, threads=True,
    )
    raw.to_parquet(CACHE_PATH)
    print('저장 완료:', CACHE_PATH)

# 종목별 유효성 검사 — 전체 구간의 95% 이상 거래일 데이터가 있는 종목만 유지
all_bdays = pd.bdate_range(TRAIN_START, TEST_END)
min_len = int(len(all_bdays) * 0.95)

symbols, dropped = [], []
per_symbol = {}
for sym in UNIVERSE:
    try:
        df = raw[sym][['Open', 'High', 'Low', 'Close']].dropna()
    except KeyError:
        dropped.append(sym)
        continue
    if len(df) < min_len:
        dropped.append(sym)
        continue
    per_symbol[sym] = df
    symbols.append(sym)

N_ASSETS = len(symbols)
print(f'유효 종목: {N_ASSETS}/{len(UNIVERSE)}' + (f' (제외: {dropped})' if dropped else ''))
assert N_ASSETS >= 30, '유효 종목이 너무 적음 — UNIVERSE 재검토 필요'


44종목 다운로드 중 (yfinance)...


[*********************100%***********************]  44 of 44 completed


저장 완료: /content/drive/MyDrive/졸업프로젝트/data/us_equity/ohlc_raw.parquet
유효 종목: 44/44
CPU times: user 3.91 s, sys: 276 ms, total: 4.19 s
Wall time: 17.7 s


## 3. 가격 행렬 + 수익률 인덱스 구성

In [4]:
%%time
# 전 종목 공통 거래일 인덱스(교집합) — 결측은 ffill(직전 종가 유지, 상장폐지 등 예외적 결측만 남음)
common_index = None
for sym in symbols:
    idx = per_symbol[sym].index
    common_index = idx if common_index is None else common_index.union(idx)
common_index = common_index.sort_values()

close_mat = np.full((N_ASSETS, len(common_index)), np.nan, dtype=np.float32)
high_mat  = np.full_like(close_mat, np.nan)
low_mat   = np.full_like(close_mat, np.nan)

for i, sym in enumerate(symbols):
    df = per_symbol[sym].reindex(common_index).ffill().bfill()
    close_mat[i] = df['Close'].to_numpy(dtype=np.float32)
    high_mat[i]  = df['High'].to_numpy(dtype=np.float32)
    low_mat[i]   = df['Low'].to_numpy(dtype=np.float32)

assert not np.isnan(close_mat).any(), '가격 행렬에 NaN 남음'
pos_of = {d: i for i, d in enumerate(common_index)}
all_dates = list(common_index)
print(f'가격 행렬: {close_mat.shape} (종목 × 거래일), {common_index[0].date()} ~ {common_index[-1].date()}')

# 1거래일 forward 수익률 인덱스 — (symbol, position) -> 다음날 수익률 (RetTarget_1d와 동일 관례)
ret_index = {}
for i, sym in enumerate(symbols):
    c = close_mat[i]
    fwd_ret = c[1:] / c[:-1] - 1
    for pos in range(len(fwd_ret)):
        ret_index[(sym, pos)] = float(fwd_ret[pos])
print(f'수익률 인덱스: {len(ret_index):,}개')


가격 행렬: (44, 2765) (종목 × 거래일), 2015-01-02 ~ 2025-12-30
수익률 인덱스: 121,616개
CPU times: user 389 ms, sys: 15.2 ms, total: 404 ms
Wall time: 433 ms


## 4. PriceWindowPortfolioEnv — EIIE/LSRE-CAAN 노트북과 완전히 동일한 환경 재사용

In [5]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class PriceWindowPortfolioEnv(gym.Env):
    '''crypto_eiie_baseline_colab.ipynb의 EIIEPortfolioEnv와 동일 로직(§0 고지 참고) — 원시 가격
    윈도우 + 직전 비중, (N+1)-dim 행동(자산 N개 + 현금 1개) -> softmax. 인덱스는 버킷이 아니라
    거래일 포지션(pos_of 값)이라는 점만 다르다.'''
    metadata = {'render_modes': []}

    def __init__(self, positions, symbols, close_mat, high_mat, low_mat,
                 window=WINDOW, tc=TC):
        super().__init__()
        self.positions = positions   # 이 에피소드가 도는 거래일 포지션 목록(전역 인덱스 기준)
        self.symbols   = symbols
        self.n_assets  = len(symbols)
        self.close_mat = close_mat
        self.high_mat  = high_mat
        self.low_mat   = low_mat
        self.window    = window
        self.tc        = tc

        obs_dim = self.n_assets * N_CHANNELS * window + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def _price_window(self, pos):
        sl = slice(pos - self.window + 1, pos + 1)
        c = self.close_mat[:, sl]
        h = self.high_mat[:, sl]
        l = self.low_mat[:, sl]
        last_close = c[:, -1:]
        last_close = np.where(last_close == 0, 1e-8, last_close)
        rc = (c / last_close).astype(np.float32)
        rh = (h / last_close).astype(np.float32)
        rl = (l / last_close).astype(np.float32)
        return np.stack([rc, rh, rl], axis=1)

    def _asset_ret(self, pos):
        rets = np.array([ret_index.get((sym, pos), 0.0) for sym in self.symbols], dtype=np.float32)
        return np.concatenate([rets, [0.0]])

    def _get_obs(self):
        pos = self.positions[self.t]
        pw = self._price_window(pos)
        return np.concatenate([pw.flatten(), self.weights[:-1]]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t                = 0
        self.weights           = np.zeros(self.n_assets + 1, dtype=np.float32)
        self.weights[-1]       = 1.0
        self.portfolio_value   = 1.0
        self.value_history     = [1.0]
        self.turnover_history  = []
        return self._get_obs(), {}

    def step(self, action):
        action = np.exp(action - action.max())
        w = (action / action.sum()).astype(np.float32)

        turnover = float(np.abs(w[:-1] - self.weights[:-1]).sum())
        tc_cost  = self.tc * turnover

        pos = self.positions[self.t]
        asset_rets = self._asset_ret(pos)
        port_ret = float((w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights = _drift_weights(w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t += 1

        done = (self.t >= len(self.positions) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover}
        return self._get_obs(), reward, done, False, info


class EIIETorchNet(nn.Module):
    '''Jiang et al.(2017) 토폴로지 — crypto_eiie_baseline_colab.ipynb §6과 동일 정의 재사용.'''

    def __init__(self, n_assets, window, n_channels=N_CHANNELS, conv1_out=2, conv2_out=20):
        super().__init__()
        self.n_assets, self.window, self.n_channels = n_assets, window, n_channels
        self.conv1 = nn.Conv1d(n_channels, conv1_out, kernel_size=3)
        conv1_len = window - 3 + 1
        self.conv2 = nn.Conv1d(conv1_out, conv2_out, kernel_size=conv1_len)
        self.score_head = nn.Linear(conv2_out + 1, 1)
        self.cash_bias = nn.Parameter(torch.zeros(1))

    def forward(self, price_window, prev_weights):
        B = price_window.shape[0]
        x = price_window.reshape(B * self.n_assets, self.n_channels, self.window)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.reshape(B, self.n_assets, -1)
        pw = prev_weights.unsqueeze(-1)
        x = torch.cat([x, pw], dim=-1)
        scores = self.score_head(x).squeeze(-1)
        cash = self.cash_bias.expand(B, 1)
        return torch.cat([scores, cash], dim=-1)


class EIIEFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, n_assets=None, window=WINDOW, n_channels=N_CHANNELS):
        features_dim = n_assets + 1
        super().__init__(observation_space, features_dim=features_dim)
        self.n_assets, self.window, self.n_channels = n_assets, window, n_channels
        self.net = EIIETorchNet(n_assets, window, n_channels)
        self._pw_size = n_assets * n_channels * window

    def forward(self, observations):
        pw_flat = observations[:, :self._pw_size]
        prev_w  = observations[:, self._pw_size:]
        pw = pw_flat.reshape(-1, self.n_assets, self.n_channels, self.window)
        return self.net(pw, prev_w)


policy_kwargs = dict(
    features_extractor_class=EIIEFeaturesExtractor,
    features_extractor_kwargs=dict(n_assets=N_ASSETS, window=WINDOW, n_channels=N_CHANNELS),
    net_arch=[],
)
print('환경·네트워크 정의 완료')


환경·네트워크 정의 완료


## 5. 날짜 분할

In [6]:
def date_range_positions(start, end):
    return [pos_of[d] for d in common_index if pd.Timestamp(start) <= d <= pd.Timestamp(end)]

train_positions = date_range_positions(TRAIN_START, TRAIN_END)
valid_positions  = date_range_positions(VALID_START, VALID_END)
test_positions   = date_range_positions(TEST_START, TEST_END)

# 윈도우 워밍업만큼은 각 구간 시작에서 확보돼 있어야 함(전역 인덱스 기준이라 자동으로 만족되지만 확인)
train_positions = [p for p in train_positions if p >= WINDOW]
valid_positions  = [p for p in valid_positions if p >= WINDOW]
test_positions   = [p for p in test_positions if p >= WINDOW]

print(f'train: {len(train_positions)}일, valid: {len(valid_positions)}일, test: {len(test_positions)}일')

_env = PriceWindowPortfolioEnv(train_positions[:200], symbols, close_mat, high_mat, low_mat)
check_env(_env, warn=True)
print('환경 유효성 검사 완료')


train: 1964일, valid: 250일, test: 501일
환경 유효성 검사 완료


## 6. §동기 확인 — 단일 정책 PPO 베이스라인 (1.1절과 동일 질문)

In [7]:
%%time
def equal_weight_backtest(positions):
    env = PriceWindowPortfolioEnv(positions, symbols, close_mat, high_mat, low_mat)
    obs, _ = env.reset()
    done = False
    action = np.zeros(env.n_assets + 1, dtype=np.float32)  # softmax(0) = 균등
    while not done:
        obs, _, done, _, info = env.step(action)
    return np.array(env.value_history), list(env.turnover_history)


def perf_row(values, turnovers, tc=TC, steps_per_year=252):
    rets   = np.diff(values) / values[:-1]
    ann    = values[-1] ** (steps_per_year / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    mdd    = ((values / np.maximum.accumulate(values)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    return {
        '총 수익률': f'{(values[-1]-1)*100:.2f}%', '연환산': f'{ann*100:.2f}%',
        'Sharpe': f'{sharpe:.3f}', 'MDD': f'{mdd*100:.2f}%',
        '평균 턴오버/스텝': f'{t.mean()*100:.2f}%',
    }


single_train_env = Monitor(PriceWindowPortfolioEnv(train_positions, symbols, close_mat, high_mat, low_mat))
single_ppo = PPO(
    'MlpPolicy', single_train_env,
    learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
    max_grad_norm=0.5, policy_kwargs=policy_kwargs,
    verbose=0, device='cpu', seed=42,
)
single_ppo.learn(total_timesteps=SINGLE_PPO_TIMESTEPS, progress_bar=True)
single_ppo.save(str(MODEL_DIR / 'single_ppo_seed42.zip'))

def run_model(model, positions):
    env = PriceWindowPortfolioEnv(positions, symbols, close_mat, high_mat, low_mat)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
    return np.array(env.value_history), list(env.turnover_history)

ew_test_vals, ew_test_turns = equal_weight_backtest(test_positions)
sp_test_vals, sp_test_turns = run_model(single_ppo, test_positions)

print('=== §동기 확인: 단일 정책 PPO vs 등가중 (test, 2024-2025) ===\n')
print('등가중:  ', perf_row(ew_test_vals, ew_test_turns))
print('단일 PPO:', perf_row(sp_test_vals, sp_test_turns))


Output()

=== §동기 확인: 단일 정책 PPO vs 등가중 (test, 2024-2025) ===

등가중:   {'총 수익률': '45.74%', '연환산': '20.90%', 'Sharpe': '1.473', 'MDD': '-16.32%', '평균 턴오버/스텝': '0.20%'}
단일 PPO: {'총 수익률': '46.20%', '연환산': '21.10%', 'Sharpe': '1.498', 'MDD': '-15.99%', '평균 턴오버/스텝': '0.38%'}
CPU times: user 8min 37s, sys: 3.34 s, total: 8min 41s
Wall time: 8min 55s


## 7. Stage A — 시장 레짐 분할 (크립토·KOSPI와 동일 코드 재사용)

In [8]:
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


market_segmentation 함수 정의 완료


In [9]:
eq_ret_per_pos = np.array([
    np.mean([ret_index.get((sym, p), 0.0) for sym in symbols])
    for p in train_positions
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_pos), index=range(len(train_positions)))

seg = segment_and_label(eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD, lowpass_window=LOWPASS_WINDOW)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.1f}일)")
print(seg.chunks['regime'].value_counts())


구간 수: 34 (평균 길이 58.7일)
regime
rally       12
pullback    10
sideways     4
bear         4
bull         4
Name: count, dtype: int64


## 8. Stage B — 레짐 선호도 기반 β풀 학습

In [10]:
class ChunkedPriceWindowEnv(PriceWindowPortfolioEnv):
    '''KOSPI §6의 ChunkedKospiPortfolioEnv와 동일한 β-우선순위 청크 샘플링.'''

    def __init__(self, all_positions, chunks, beta, risk_threshold=0.2, rng=None, **env_kwargs):
        self.all_positions = all_positions
        self.chunks         = chunks.reset_index(drop=True)
        self.beta           = beta
        self.priority        = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng             = rng or np.random.default_rng()
        super().__init__(positions=all_positions[:2], **env_kwargs)

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.positions = self.all_positions[int(row['start_idx']):int(row['end_idx']) + 1]
        if len(self.positions) < WINDOW + 2:
            self.positions = self.all_positions[max(0, int(row['start_idx']) - WINDOW):int(row['end_idx']) + 2]
        return super().reset(seed=seed, options=options)


env_kwargs = dict(symbols=symbols, close_mat=close_mat, high_mat=high_mat, low_mat=low_mat)

_test_env = ChunkedPriceWindowEnv(all_positions=train_positions, chunks=seg.chunks, beta=90,
                                    rng=np.random.default_rng(0), **env_kwargs)
for _ in range(3):
    _test_env.reset()
    print(f'β=90 샘플 구간: {len(_test_env.positions)}일')


β=90 샘플 구간: 84일
β=90 샘플 구간: 75일
β=90 샘플 구간: 72일


In [11]:
%%time
pool_models = {}

for pool_i, beta in enumerate(POOL_BETAS):
    print(f'\n=== β={beta} 에이전트 학습 ({POOL_TIMESTEPS_EACH:,} 스텝) ===')
    rng = np.random.default_rng(1000 + pool_i)
    chunk_env = Monitor(ChunkedPriceWindowEnv(
        all_positions=train_positions, chunks=seg.chunks, beta=beta,
        risk_threshold=RISK_THRESHOLD, rng=rng, **env_kwargs,
    ))
    model = PPO(
        'MlpPolicy', chunk_env,
        learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
        gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
        max_grad_norm=0.5, policy_kwargs=policy_kwargs,
        verbose=0, device='cpu', seed=42,
    )
    model.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)

    save_path = MODEL_DIR / f'agent_beta_{beta}.zip'
    model.save(str(save_path))
    pool_models[beta] = model
    print(f'저장: {save_path.name}')

print(f'\n풀 학습 완료: {len(pool_models)}개 에이전트')


Output()


=== β=-90 에이전트 학습 (100,000 스텝) ===


Output()

저장: agent_beta_-90.zip

=== β=-30 에이전트 학습 (100,000 스텝) ===


Output()

저장: agent_beta_-30.zip

=== β=30 에이전트 학습 (100,000 스텝) ===


Output()

저장: agent_beta_30.zip

=== β=90 에이전트 학습 (100,000 스텝) ===


저장: agent_beta_90.zip

풀 학습 완료: 4개 에이전트
CPU times: user 34min 28s, sys: 12.5 s, total: 34min 41s
Wall time: 35min 25s


In [12]:
# 재실행용 로더 (idempotent)
if 'pool_models' not in globals() or not pool_models:
    pool_models = {b: PPO.load(str(MODEL_DIR / f'agent_beta_{b}.zip'),
                                 custom_objects={'policy_kwargs': policy_kwargs})
                    for b in POOL_BETAS}
    print('풀 에이전트 로드 완료:', list(pool_models.keys()))


## 9. 다양성 진단 — top-10 보유종목 중첩도 + 비중 상관

In [13]:
def run_solo_raw(model, positions):
    env = PriceWindowPortfolioEnv(positions, symbols, close_mat, high_mat, low_mat)
    obs, _ = env.reset()
    done = False
    weight_history = [env.weights.copy()]
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
        weight_history.append(env.weights.copy())
    return np.array(env.value_history), np.array(weight_history), list(env.turnover_history)


print('=== [§9] β별 valid 구간 비중 궤적으로 다양성 진단 ===\n')
weight_traj, top10_sets = {}, {}
for beta in POOL_BETAS:
    _, w_hist, turns = run_solo_raw(pool_models[beta], valid_positions)
    weight_traj[beta] = w_hist
    top10_per_step = [set(np.argsort(w_hist[t, :-1])[-10:]) for t in range(len(w_hist))]
    ever_top10 = set().union(*top10_per_step)
    top10_sets[beta] = ever_top10
    print(f'β={beta:>4d}: 평균 턴오버/스텝 = {np.mean(turns)*100:.2f}%, '
          f'valid 구간 top-10 순환 종목 수 = {len(ever_top10)}/{N_ASSETS}')

print('\n--- β쌍별 top-10 보유종목 Jaccard 중첩도(낮을수록 다양) ---')
betas = POOL_BETAS
for i, b1 in enumerate(betas):
    for b2 in betas[i + 1:]:
        inter = len(top10_sets[b1] & top10_sets[b2])
        union = len(top10_sets[b1] | top10_sets[b2])
        print(f'  β={b1:>4d} vs β={b2:>4d}: Jaccard = {inter/union:.3f}')

print('\n--- β쌍별 비중 궤적 평균 상관(자산별 시계열 상관의 평균) ---')
corr_rows = {}
for b1 in betas:
    row = {}
    for b2 in betas:
        corrs = [np.corrcoef(weight_traj[b1][:, a], weight_traj[b2][:, a])[0, 1] for a in range(N_ASSETS + 1)]
        row[f'β={b2}'] = np.nanmean(corrs)
    corr_rows[f'β={b1}'] = row
corr_df = pd.DataFrame(corr_rows).T
print(corr_df.to_string(float_format=lambda x: f'{x:.3f}'))


=== [§9] β별 valid 구간 비중 궤적으로 다양성 진단 ===

β= -90: 평균 턴오버/스텝 = 0.73%, valid 구간 top-10 순환 종목 수 = 19/44
β= -30: 평균 턴오버/스텝 = 0.51%, valid 구간 top-10 순환 종목 수 = 19/44
β=  30: 평균 턴오버/스텝 = 0.56%, valid 구간 top-10 순환 종목 수 = 20/44
β=  90: 평균 턴오버/스텝 = 0.68%, valid 구간 top-10 순환 종목 수 = 21/44

--- β쌍별 top-10 보유종목 Jaccard 중첩도(낮을수록 다양) ---
  β= -90 vs β= -30: Jaccard = 0.727
  β= -90 vs β=  30: Jaccard = 0.625
  β= -90 vs β=  90: Jaccard = 0.538
  β= -30 vs β=  30: Jaccard = 0.696
  β= -30 vs β=  90: Jaccard = 0.600
  β=  30 vs β=  90: Jaccard = 0.708

--- β쌍별 비중 궤적 평균 상관(자산별 시계열 상관의 평균) ---
       β=-90  β=-30  β=30  β=90
β=-90  1.000  0.883 0.910 0.861
β=-30  0.883  1.000 0.977 0.888
β=30   0.910  0.977 1.000 0.905
β=90   0.861  0.888 0.905 1.000


### 9-B. 판정 기준

1. top-10 Jaccard 중첩도가 β쌍 대부분에서 낮고(예: <0.7), 비중 상관도 대각선 제외 대부분 1에
   가깝지 않으면 다양성이 있다고 본다 — 크립토 v3(전면 붕괴)·KOSPI(부분 붕괴, 극단 β쌍만 수렴)와
   비교해 어느 패턴에 가까운지 판정한다.
2. 결과는 `paper/GAP_ANALYSIS.md`·`FINDINGS_LOG.md`·`DRAFT.md`/`DRAFT_KR.md` 7장에 KOSPI 결과와
   나란히 반영한다.

## 10. 추론 시점 관성 스무딩 — β별 α 스윕 (valid에서 선택)

In [14]:
def run_solo_smoothed(model, alpha, positions):
    '''PriceWindowPortfolioEnv.step()도 내부에서 softmax(action)을 적용하므로 크립토 §9/KOSPI §8과
    동일한 log-trick으로 재학습 없이 w_new=(1-α)w_old+α·softmax(raw_action)를 반영한다.'''
    env = PriceWindowPortfolioEnv(positions, symbols, close_mat, high_mat, low_mat)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        raw = np.exp(a - a.max())
        raw = raw / raw.sum()
        w = (1 - alpha) * env.weights + alpha * raw
        obs, _, done, _, info = env.step(np.log(np.maximum(w, 1e-12)))
    return np.array(env.value_history), list(env.turnover_history)


print('=== [§10] β×α 스윕 — valid(2023)에서 최선의 조합 탐색 ===\n')
sweep_rows = {}
best_by_beta = {}
for beta in POOL_BETAS:
    best_val, best_alpha = -np.inf, None
    for alpha in SMOOTH_ALPHAS:
        v_vals, v_turns = run_solo_smoothed(pool_models[beta], alpha, valid_positions)
        row = perf_row(v_vals, v_turns)
        sweep_rows[f'β={beta}, α={alpha}'] = row
        rets = np.diff(v_vals) / v_vals[:-1]
        sharpe_val = rets.mean() / (rets.std() + 1e-8) * np.sqrt(252)
        if sharpe_val > best_val:
            best_val, best_alpha = sharpe_val, alpha
    best_by_beta[beta] = (best_alpha, best_val)
    print(f'β={beta}: valid 최선 α={best_alpha} (Sharpe={best_val:.3f})')

sweep_df = pd.DataFrame(sweep_rows).T
print()
print(sweep_df.to_string())
sweep_df.to_csv(DATA_DIR / 'us_equity_hrl_alpha_sweep_valid.csv', encoding='utf-8-sig')
print('\n저장 완료: us_equity_hrl_alpha_sweep_valid.csv')


=== [§10] β×α 스윕 — valid(2023)에서 최선의 조합 탐색 ===

β=-90: valid 최선 α=1.0 (Sharpe=1.553)
β=-30: valid 최선 α=1.0 (Sharpe=1.521)
β=30: valid 최선 α=1.0 (Sharpe=1.511)
β=90: valid 최선 α=1.0 (Sharpe=1.480)

                총 수익률     연환산 Sharpe      MDD 평균 턴오버/스텝
β=-90, α=1.0   18.83%  19.08%  1.553  -10.22%     0.73%
β=-90, α=0.3   17.60%  17.83%  1.483  -10.22%     0.58%
β=-90, α=0.15  16.53%  16.74%  1.424  -10.22%     0.54%
β=-90, α=0.05  15.11%  15.31%  1.396  -10.20%     0.46%
β=-90, α=0.03  14.39%  14.58%  1.419  -10.15%     0.43%
β=-90, α=0.02  13.48%  13.65%  1.438   -9.91%     0.41%
β=-90, α=0.01  11.00%  11.14%  1.471   -8.56%     0.36%
β=-30, α=1.0   18.87%  19.12%  1.521  -10.22%     0.51%
β=-30, α=0.3   17.52%  17.75%  1.444  -10.22%     0.44%
β=-30, α=0.15  16.34%  16.56%  1.379  -10.22%     0.43%
β=-30, α=0.05  14.84%  15.03%  1.340  -10.22%     0.40%
β=-30, α=0.03  14.20%  14.38%  1.367  -10.17%     0.40%
β=-30, α=0.02  13.40%  13.58%  1.394   -9.94%     0.39%
β=-30, α=0.01  11.09%

## 11. 최종 비교 — valid에서 고른 β/α를 test에서 1회 실행

In [15]:
FINAL_BETA  = max(best_by_beta, key=lambda b: best_by_beta[b][1])
FINAL_ALPHA = best_by_beta[FINAL_BETA][0]
print(f'최종 후보: β={FINAL_BETA}, α={FINAL_ALPHA} (valid Sharpe={best_by_beta[FINAL_BETA][1]:.3f})\n')

print('=== [§11] 최종 비교 — valid(2023) / test(2024-2025, "1회만" 실행) ===\n')

v_final_vals, v_final_turns = run_solo_smoothed(pool_models[FINAL_BETA], FINAL_ALPHA, valid_positions)
t_final_vals, t_final_turns = run_solo_smoothed(pool_models[FINAL_BETA], FINAL_ALPHA, test_positions)
ew_valid_vals, ew_valid_turns = equal_weight_backtest(valid_positions)
# ew_test_vals, ew_test_turns, sp_test_vals, sp_test_turns computed in §6 already

comparison_valid = {
    '등가중 벤치마크':                             perf_row(ew_valid_vals, ew_valid_turns),
    f'미국주식 HRL β={FINAL_BETA}, α={FINAL_ALPHA}': perf_row(v_final_vals, v_final_turns),
}
comparison_test = {
    '등가중 벤치마크':                             perf_row(ew_test_vals, ew_test_turns),
    '단일 정책 PPO (§6, 참고)':                    perf_row(sp_test_vals, sp_test_turns),
    f'미국주식 HRL β={FINAL_BETA}, α={FINAL_ALPHA}': perf_row(t_final_vals, t_final_turns),
}

valid_df = pd.DataFrame(comparison_valid).T
test_df  = pd.DataFrame(comparison_test).T

print('--- valid(2023) ---')
print(valid_df.to_string())
print('\n--- test(2024-2025) ---')
print(test_df.to_string())

valid_df.to_csv(DATA_DIR / 'us_equity_hrl_final_valid.csv', encoding='utf-8-sig')
test_df.to_csv(DATA_DIR / 'us_equity_hrl_final_test.csv', encoding='utf-8-sig')
print('\n저장 완료: us_equity_hrl_final_{valid,test}.csv')


최종 후보: β=-90, α=1.0 (valid Sharpe=1.553)

=== [§11] 최종 비교 — valid(2023) / test(2024-2025, "1회만" 실행) ===

--- valid(2023) ---
                        총 수익률     연환산 Sharpe      MDD 평균 턴오버/스텝
등가중 벤치마크               18.81%  19.06%  1.525  -10.29%     0.39%
미국주식 HRL β=-90, α=1.0  18.83%  19.08%  1.553  -10.22%     0.73%

--- test(2024-2025) ---
                        총 수익률     연환산 Sharpe      MDD 평균 턴오버/스텝
등가중 벤치마크               45.74%  20.90%  1.473  -16.32%     0.20%
단일 정책 PPO (§6, 참고)     46.20%  21.10%  1.498  -15.99%     0.38%
미국주식 HRL β=-90, α=1.0  44.95%  20.57%  1.479  -16.23%     0.52%

저장 완료: us_equity_hrl_final_{valid,test}.csv


## 12. 판정 기준

1. **1차(핵심)**: test에서 미국주식 HRL이 등가중을 이기는지, valid Sharpe와 test Sharpe 격차가
   크지 않은지(§4.6/§16/KOSPI §9와 동일 진단) 확인한다. 격차가 크면(특히 valid만 좋고 test가
   나쁘면 KOSPI와 동일 패턴, 반대면 크립토 β=30과 동일 패턴) 기각한다.
2. **2차**: 단일 정책 PPO(§6)를 확실히 앞서는지 확인 — 이기지 못하면 "계층 구조가 이 세팅에서도
   도움 안 됨"이라는 부정적이지만 그 자체로 보고 가치 있는 결과.
3. **§9의 다양성 진단과 함께 읽을 것**: 다양성이 붕괴했다면 여기서 나온 성과 차이가 "레짐 조건부
   학습의 진짜 효과"가 아닐 수 있다.
4. **중요한 주의**: 이 노트북은 임베딩 백본 없이 원시 가격 윈도우만 쓰므로, 여기서 나온 성과를
   크립토(β=-30, +5.66%)나 KOSPI 결과와 **절대 수치로 직접 비교하지 말 것** — 비교 가능한 것은
   오직 "이 노트북 안에서 HRL이 등가중을 이기는가"와 "valid/test 격차가 작은가"라는 두 가지
   상대적 진단뿐이다.
5. 결과는 `paper/DRAFT.md`/`DRAFT_KR.md` 7장에 KOSPI 결과와 나란히, `paper/GAP_ANALYSIS.md`·
   `FINDINGS_LOG.md`에 반영한다.